# 05 | Search attention and competing explanations

**Author: Chanakya**

Use relative search interest to challenge the growth narrative. These are search terms in India, not topics, unique viewers or subscription attribution. Separate requests have separate normalization.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='05_search_attention'
shared.ACTIVE_SOURCES=['google_trends_daily_table_v2', 'google_trends_football_table_v5', 'google_trends_query_context_v2', 'v4_trends_2025_timeline']

Offline inputs: raw-v5-2026-09-14 | Author: Chanakya


## 1. Reconstruct the saved query tables
The 2026 four-term request shares one scale. The football pair and 2025 weekly pair do not share that scale. Preserve query context, rounding and boundary weeks. No extrapolation to absolute searches.

In [2]:
daily=pd.read_csv(path('google_trends_daily_table_v2'),sep='\t');daily.columns=['date','tennis','FanCode','Formula 1','MotoGP'];daily['date']=pd.to_datetime(daily.date)
football=pd.read_csv(path('google_trends_football_table_v5'),sep='\t');football.columns=['date','football','FanCode'];football['date']=pd.to_datetime(football.date)
s=text('v4_trends_2025_timeline');weeklyraw=json.loads(s[s.index('{'):])['default']['timelineData']
weekly=pd.DataFrame([dict(week=pd.to_datetime(int(x['time']),unit='s',utc=True),label=x['formattedTime'],tennis=x['value'][0],FanCode=x['value'][1]) for x in weeklyraw])
table(daily,'trends_daily_four_terms',True);table(football,'trends_football_pair',True);table(weekly,'trends_2025_weekly',True)
print(text('google_trends_query_context_v2'))
figx,axes=plt.subplots(2,1,figsize=(11,6))
for col in daily.columns[1:]:axes[0].plot(daily.date,daily[col].rolling(7,min_periods=7).mean(),label=col)
axes[0].set(title='Within-request 7-day mean search interest',ylabel='Relative interest');axes[0].legend(ncol=4,fontsize=8)
axes[1].plot(football.date,football.football,label='football');axes[1].plot(football.date,football.FanCode,label='FanCode');axes[1].set(title='Separate football pair: different scale and rounding',ylabel='Relative interest');axes[1].legend();fig('05_search_series','Google Trends India web search, Jan1–Sep12 2026. Context contains noisy related queries. No subscriber inference.')

tennis | Search term
FanCode | Search term
Formula 1 | Search term
MotoGP | Search term
India
1/1/26 - 9/12/26
All categories
Web Search
Related queries: Rising
Tennis:
posts category tennis latestsportsbuzz | Breakout
tennis news on latestsportsbuzz | Breakout
sportsmag india | Breakout
sportsmag.in hockey | Breakout
sportsmag.in football | Breakout
FanCode:
ind vs zim live | Breakout
india zimbabwe t20 live | Breakout
ind vs zim live streaming | Breakout
ind vs zimbabwe | Breakout
fancode isl | Breakout
Formula 1:
kitkat formula 1 chocolate | Breakout
formula 1 kitkat price | Breakout
formula 1 kitkat | Breakout
walmart near me | Breakout
indian bike driving 3d formula 1 car cheat code | +3,650%
MotoGP:
2026 motogp world championship | Breakout
bet365 motogp | Breakout
tissot motogp limited edition 2026 | Breakout
motogp 2026 wiki | Breakout
fabio di giannantonio | Breakout



<Figure size 1100x600 with 2 Axes>

## 2. Association and a temporal placebo check
Report rank correlation in levels and first differences. Circular shifts preserve each series’ shape but displace its alignment. The comparison is a sensitivity diagnostic, not a valid randomization p-value under guaranteed exchangeability. Avoid selecting a preferred lag after inspecting outcomes.

In [3]:
from scipy.stats import spearmanr
assoc=[]
for label,df,cols in [('four_term',daily,['tennis','Formula 1','MotoGP']),('football_pair',football,['football'])]:
 for sport in cols:
  x=df[sport].to_numpy();y=df.FanCode.to_numpy();rho=spearmanr(x,y).statistic;diff=spearmanr(np.diff(x),np.diff(y)).statistic
  shifted=np.array([spearmanr(x,np.roll(y,k)).statistic for k in range(8,len(y)-7)])
  assoc.append(dict(request=label,sport=sport,level_rho=rho,difference_rho=diff,circular_shift_abs_exceedance=float((np.abs(shifted)>=abs(rho)).mean()),zero_share=float((x==0).mean()),n=len(x)))
assoc=pd.DataFrame(assoc);display(table(assoc,'05_search_associations'))
plt.figure(figsize=(8,4));x=np.arange(len(assoc));plt.bar(x-.17,assoc.level_rho,.34,label='Levels');plt.bar(x+.17,assoc.difference_rho,.34,label='First differences');plt.xticks(x,assoc.sport);plt.axhline(0,color='black',lw=.8);plt.ylabel('Spearman correlation with FanCode');plt.title('Associations depend on the request and time-series treatment');plt.legend();fig('05_search_associations','Descriptive correlations. Broad terms and cricket/other events confound interpretation. No causal sport contribution.')
peaks=daily.nlargest(15,'FanCode').copy();f1=read('f1_races');fdates=pd.to_datetime(f1.start_utc,utc=True).dt.tz_convert('Asia/Kolkata').dt.tz_localize(None).dt.normalize();slots=read('atp_slots');adates=pd.to_datetime(slots.date_ist)
peaks['days_to_nearest_F1_race']=[min(abs((fdates-date).dt.days)) for date in peaks.date];peaks['days_to_nearest_sampled_ATP_slot']=[min(abs((adates-date).dt.days)) for date in peaks.date];display(table(peaks,'05_peak_date_context'))
zero=pd.DataFrame([dict(request='football pair',term='FanCode',zero_days=int((football.FanCode==0).sum()),days=len(football)),dict(request='four term',term='FanCode',zero_days=int((daily.FanCode==0).sum()),days=len(daily))]);display(table(zero,'05_rounding_diagnostic'))
check('05_trends',{'daily_dates_255':len(daily)==255 and daily.date.is_unique,'football_dates_identical':daily.date.equals(football.date),'weekly_bins_53':len(weekly)==53,'bounded_values':bool(daily.iloc[:,1:].apply(lambda c:c.between(0,100).all()).all())})
report('05_search_findings','Google Trends cannot independently verify which sport drives paid subscriptions. Noisy related queries and request-level rounding materially constrain interpretation. Use the series for timing exploration and creative hypotheses, with observed peak dates explicitly separated from causal explanations. The 2025 pair provides historical within-request seasonality only, not a comparable YoY level.')

,request,sport,level_rho,difference_rho,circular_shift_abs_exceedance,zero_share,n
0,four_term,tennis,0.064,0.116,0.500,0.000,255
1,four_term,Formula 1,0.458,0.367,0.000,0.000,255
2,four_term,MotoGP,0.174,0.271,0.108,0.008,255
3,football_pair,football,0.179,0.193,0.000,0.000,255


<Figure size 800x400 with 1 Axes>

,date,tennis,FanCode,Formula 1,MotoGP,days_to_nearest_F1_race,days_to_nearest_sampled_ATP_slot
203,2026-07-23,18,93,6,1,3,32
205,2026-07-25,21,61,9,1,1,34
206,2026-07-26,25,57,15,1,0,34
10,2026-01-11,25,39,7,1,56,0
73,2026-03-15,33,30,21,2,0,1
80,2026-03-22,28,21,8,8,7,6
65,2026-03-07,26,17,15,2,1,7
66,2026-03-08,25,17,21,1,0,8
129,2026-05-10,28,16,7,3,6,7
44,2026-02-14,38,15,8,1,22,1


,request,term,zero_days,days
0,football pair,FanCode,188,255
1,four term,FanCode,0,255


,check,passed
0,daily_dates_255,True
1,football_dates_identical,True
2,weekly_bins_53,True
3,bounded_values,True


## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [4]:
references=source_table(['google_trends_daily_table_v2', 'google_trends_football_table_v5', 'google_trends_query_context_v2', 'v4_trends_2025_timeline'])
display(table(references,'05_source_references'))

,source_id,url,raw_file,retrieved
0,google_trends_daily_table_v2,https://trends.google.com/trends/explore?date=...,data/raw/search_interest/google_trends_daily_t...,2026-09-14T06:14:12.972022+00:00
1,google_trends_football_table_v5,https://trends.google.com/trends/explore?date=...,data/raw/search_interest/google_trends_footbal...,2026-09-14T16:59:45.440909+00:00
2,google_trends_query_context_v2,https://trends.google.com/trends/explore?date=...,data/raw/search_interest/google_trends_query_c...,2026-09-14T06:14:12.972022+00:00
3,v4_trends_2025_timeline,https://trends.google.com/trends/api/widgetdat...,data/raw/search_interest/v4_trends_2025_timeli...,2026-09-14T16:42:41.067672+00:00
